# Hybrid CNN-Transformer Crowd Counting on ShanghaiTech

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RashmiNair2021/Flask_basic/blob/claude/cnn-transformer-crowd-counting-6h4tom/colab/ShanghaiTech_CNN_Transformer_Crowd_Counting.ipynb)

An end-to-end, GPU-ready pipeline for crowd counting on the **ShanghaiTech** dataset (Part A or Part B),
built around a hybrid architecture that combines:

- **CNN backbone** (VGG16 transfer learning) for local feature extraction
- **Multi-scale feature fusion** (mini feature-pyramid across two backbone depths)
- **Dilated Pyramid Pooling Module (DPPM)** — an ASPP-style block combining *dilated convolutions*
  at multiple rates with a global-pooling *pyramid* branch for multi-scale context
- **Spatial Transformer Network (STN)** to learn perspective/scale-invariant feature warping
- **Transformer encoder** (multi-head self-attention + 2D positional encoding) for long-range,
  global crowd context
- **Graph-based occlusion reasoning module** — a lightweight GNN that lets occluded/ambiguous
  regions borrow evidence from visually-similar regions elsewhere in the scene
- **Image patching** for both training (random-crop patches) and inference (tiled sliding-window),
  plus **mixed precision**, **gradient accumulation**, **backbone freezing**, and **gradient
  checkpointing** for memory efficiency
- **Adaptive learning rate** (`ReduceLROnPlateau`, driven by validation MAE)
- **Best-model checkpointing** — compares every epoch against the best validation result so far
- **Train / validation split** with live loss & MAE (the regression analogue of "accuracy") curves
- A **combined loss**: pixel-wise MSE + SSIM (structural) + a direct counting L1 term — a
  combination shown in crowd-counting literature to be more effective than any single term alone

> Runtime: **Runtime → Change runtime type → GPU (T4/A100)** before executing.


## 1. Environment setup

Only a couple of packages are missing from the default Colab image. Everything else
(`torch`, `torchvision`, `numpy`, `scipy`, `matplotlib`, `opencv`, `tqdm`) ships pre-installed.


In [ ]:
!pip -q install kagglehub h5py --upgrade


In [ ]:
import os
import math
import glob
import json
import random
import zipfile
import shutil
from dataclasses import dataclass, asdict

import numpy as np
from PIL import Image
import scipy.io as sio
from scipy.spatial import KDTree

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

import torchvision.transforms as T
import torchvision.models as tv_models

import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

# bfloat16 has the same exponent range as float32 (no overflow risk from squaring/summing
# already-scaled density values, unlike float16), so prefer it whenever the GPU supports it.
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print("Mixed-precision dtype:", AMP_DTYPE)


## 2. Configuration

Every tunable hyperparameter lives in one place. `patch_size` controls both the random-crop
size used during training *and* the tile size used for sliding-window inference — keeping the
transformer's token count constant and memory bounded regardless of the original image size.


In [ ]:
@dataclass
class CFG:
    # ---- dataset ----
    part: str = "A"                      # "A", "B", or "both" (combine both parts for
                                          # more training data -- see the caveat printed
                                          # in Section 3 about their differing domains)
    dataset_search_roots: tuple = (
        "/content/ShanghaiTech", "/content/shanghaitech",
        "/content/drive/MyDrive/ShanghaiTech",
    )
    density_cache_root: str = "/content/density_cache"
    val_split: float = 0.15              # held out from the official *training* split; kept
                                          # larger than a typical 90/10 split because ShanghaiTech
                                          # per-image counts vary hugely (~100s to ~1500+), so a
                                          # small val set makes the epoch-to-epoch MAE too noisy
                                          # for the scheduler/checkpointing to read reliably

    # ---- image / patch handling ----
    patch_size: int = 256                # multiple of output_stride; used for train crops AND tiled inference
    output_stride: int = 8               # backbone downsampling factor (stride of the density head)
    density_scale: float = 100.0         # scales tiny per-pixel density values for stable gradients

    # ---- model ----
    pretrained_backbone: bool = True
    freeze_backbone_stage1: bool = True  # freeze first conv block of VGG16 (memory + fewer params to update)
    transformer_layers: int = 2
    transformer_heads: int = 4
    transformer_ff_dim: int = 512
    graph_grid_size: int = 16
    graph_k: int = 8
    use_grad_checkpoint: bool = False    # flip on if you hit OOM on deep backbones

    # ---- optimization ----
    batch_size: int = 8
    num_workers: int = 2
    base_lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 120
    grad_accum_steps: int = 1
    amp: bool = True                     # automatic mixed precision
    grad_clip_norm: float = 5.0          # caps gradient norm to prevent exploding-gradient NaNs

    # ---- loss weights ----
    lambda_mse: float = 1.0
    lambda_ssim: float = 0.1
    lambda_count: float = 0.02

    # ---- scheduler ----
    lr_factor: float = 0.7               # gentler decay than 0.5: a noisy val MAE signal
                                          # shouldn't be able to crush the LR in a couple of hits
    lr_patience: int = 12                # epochs to wait for improvement before decaying;
                                          # 5 was too short given how noisy per-epoch val MAE is
                                          # on a small, high-variance validation set
    lr_min: float = 1e-6

    # ---- checkpoints ----
    ckpt_dir: str = "/content/checkpoints"


cfg = CFG()
os.makedirs(cfg.ckpt_dir, exist_ok=True)
os.makedirs(cfg.density_cache_root, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2, default=str))


## 3. Get the ShanghaiTech dataset

Pick **one** of the cells below. The Kaggle route (`tthien/shanghaitech`) is the easiest inside
Colab — it needs your `kaggle.json` API token (Kaggle account → *Settings* → *Create New Token*).

Set `cfg.part` in Section 2 to `"A"`, `"B"`, or `"both"`. The `tthien/shanghaitech` Kaggle dataset already contains both `part_A` and `part_B`, so `"both"` needs no extra download -- it just uses more of what's already there.


In [ ]:
# --- Option A: Kaggle (recommended) ------------------------------------------------
# Upload your kaggle.json first:
# from google.colab import files
# files.upload()  # select kaggle.json
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
# os.chmod("/root/.kaggle/kaggle.json", 0o600)

DATASET_PATH = None
try:
    import kagglehub
    DATASET_PATH = kagglehub.dataset_download("tthien/shanghaitech")
    print("Downloaded to:", DATASET_PATH)
except Exception as e:
    print("Kaggle download unavailable/failed:", e)
    print("Use Option B (Google Drive) or Option C (manual upload) below instead.")


In [ ]:
# --- Option B: Google Drive (if you already have a ShanghaiTech.zip in Drive) ------
# from google.colab import drive
# drive.mount('/content/drive')
# zip_path = "/content/drive/MyDrive/ShanghaiTech.zip"
# with zipfile.ZipFile(zip_path, 'r') as zf:
#     zf.extractall("/content/ShanghaiTech")
# DATASET_PATH = "/content/ShanghaiTech"


In [ ]:
# --- Option C: manual upload of a zip file -----------------------------------------
# from google.colab import files
# uploaded = files.upload()  # select your ShanghaiTech.zip
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name, 'r') as zf:
#     zf.extractall("/content/ShanghaiTech")
# DATASET_PATH = "/content/ShanghaiTech"


In [ ]:
def find_dataset_root(search_roots):
    candidates = [r for r in search_roots if r and os.path.isdir(r)]
    for root in candidates:
        for dirpath, dirnames, _ in os.walk(root):
            if "part_A" in dirnames or "part_B" in dirnames:
                return dirpath
    # fall back: look for any directory that itself is named part_A/part_B
    for root in candidates:
        for dirpath, dirnames, _ in os.walk(root):
            if os.path.basename(dirpath) in ("part_A", "part_B"):
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not locate part_A / part_B under: " + ", ".join(candidates)
        + ". Run one of the download options above first."
    )


def resolve_part_dirs(dataset_root, part):
    part_dir = os.path.join(dataset_root, f"part_{part}")
    train_img_dir = os.path.join(part_dir, "train_data", "images")
    train_gt_dir = os.path.join(part_dir, "train_data", "ground-truth")
    test_img_dir = os.path.join(part_dir, "test_data", "images")
    test_gt_dir = os.path.join(part_dir, "test_data", "ground-truth")
    for d in (train_img_dir, train_gt_dir, test_img_dir, test_gt_dir):
        assert os.path.isdir(d), f"Missing expected directory: {d}"
    return train_img_dir, train_gt_dir, test_img_dir, test_gt_dir


search_roots = list(cfg.dataset_search_roots)
if DATASET_PATH:
    search_roots = [DATASET_PATH] + search_roots

DATASET_ROOT = find_dataset_root(search_roots)
print("Dataset root:", DATASET_ROOT)

active_parts = ["A", "B"] if cfg.part == "both" else [cfg.part]

# Per-part image lists and ground-truth dirs -- kept separate (rather than merged into one
# flat list) because Part A uses geometry-adaptive Gaussians and Part B uses a fixed sigma
# (see precompute_density_maps in Section 4), and each part's images live in their own
# density-cache directory.
train_image_paths_by_part, test_image_paths_by_part = {}, {}
train_gt_dir_by_part, test_gt_dir_by_part = {}, {}
for p in active_parts:
    train_img_dir, train_gt_dir, test_img_dir, test_gt_dir = resolve_part_dirs(DATASET_ROOT, p)
    train_image_paths_by_part[p] = sorted(glob.glob(os.path.join(train_img_dir, "*.jpg")))
    test_image_paths_by_part[p] = sorted(glob.glob(os.path.join(test_img_dir, "*.jpg")))
    train_gt_dir_by_part[p] = train_gt_dir
    test_gt_dir_by_part[p] = test_gt_dir
    print(f"part_{p}: {len(train_image_paths_by_part[p])} train images, "
          f"{len(test_image_paths_by_part[p])} test images")

if cfg.part == "both":
    print(
        "\nNote: Part A (dense, web-crawled, avg ~501 people/image, variable resolution) and "
        "Part B (sparser, fixed surveillance camera, avg ~123 people/image) are different "
        "domains -- the literature always reports them as separate benchmarks, never pooled. "
        "Combining them gives the model roughly 2x the images and more viewpoint diversity, at "
        "the cost of a more heterogeneous training signal. Section 11 reports test MAE/RMSE "
        "per part as well as combined, so you can check whether either part is hurt by mixing."
    )


## 4. Ground-truth parsing & geometry-adaptive density maps

ShanghaiTech ships point annotations (one dot per head) in `.mat` files. We convert each
image's point set into a continuous **density map** by splashing a Gaussian at every head
location. Following MCNN/CSRNet, the Gaussian spread (`sigma`) is **geometry-adaptive**: it is
derived from the average distance to each point's nearest neighbours, so dense clusters get
tight kernels and sparse regions get wider ones.

To keep this fast, each Gaussian is rendered into a small local window (sized to its own
`sigma`) rather than blurring the entire image per point — this is itself a memory/CPU
optimisation for a step that would otherwise dominate preprocessing time on the crowded
Part A images.

The resulting maps are cached to disk as `.npy` files so this expensive step runs only once.


In [ ]:
def load_gt_points(mat_path):
    mat = sio.loadmat(mat_path)
    points = mat["image_info"][0, 0][0, 0][0]
    return points.astype(np.float32)


def _gaussian_kernel_2d(sigma):
    radius = max(1, int(3 * sigma))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    kernel /= kernel.sum() + 1e-12
    return kernel, radius


def generate_density_map(img_hw, points, adaptive=True, k=3, beta=0.3,
                          fixed_sigma=15.0, max_sigma=20.0):
    h, w = img_hw
    density = np.zeros((h, w), dtype=np.float32)
    n = points.shape[0]
    if n == 0:
        return density

    if adaptive and n > k:
        tree = KDTree(points)
        dists, _ = tree.query(points, k=k + 1)
        sigmas = dists[:, 1:].mean(axis=1) * beta
        sigmas = np.clip(sigmas, 1.0, max_sigma)
    else:
        sigmas = np.full(n, fixed_sigma, dtype=np.float32)

    for i in range(n):
        x = int(round(float(points[i, 0])))
        y = int(round(float(points[i, 1])))
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        kernel, radius = _gaussian_kernel_2d(sigmas[i])
        x0, x1 = x - radius, x + radius + 1
        y0, y1 = y - radius, y + radius + 1
        kx0, kx1 = max(0, -x0), kernel.shape[1] - max(0, x1 - w)
        ky0, ky1 = max(0, -y0), kernel.shape[0] - max(0, y1 - h)
        cx0, cx1 = max(0, x0), min(w, x1)
        cy0, cy1 = max(0, y0), min(h, y1)
        density[cy0:cy1, cx0:cx1] += kernel[ky0:ky1, kx0:kx1]
    return density


def precompute_density_maps(image_paths, gt_dir, out_dir, part="A"):
    os.makedirs(out_dir, exist_ok=True)
    adaptive = (part == "A")
    for img_path in tqdm(image_paths, desc=f"Density maps -> {out_dir}"):
        fname = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(out_dir, fname + ".npy")
        if os.path.exists(out_path):
            continue
        gt_path = os.path.join(gt_dir, "GT_" + fname + ".mat")
        points = load_gt_points(gt_path)
        with Image.open(img_path) as im:
            w, h = im.size
        density = generate_density_map((h, w), points, adaptive=adaptive)
        np.save(out_path, density.astype(np.float32))


def load_density_map(density_dir, fname):
    return np.load(os.path.join(density_dir, fname + ".npy"))


In [ ]:
train_density_dir_by_part, test_density_dir_by_part = {}, {}
for p in active_parts:
    train_density_dir_by_part[p] = os.path.join(cfg.density_cache_root, f"part_{p}_train")
    test_density_dir_by_part[p] = os.path.join(cfg.density_cache_root, f"part_{p}_test")
    precompute_density_maps(train_image_paths_by_part[p], train_gt_dir_by_part[p],
                             train_density_dir_by_part[p], part=p)
    precompute_density_maps(test_image_paths_by_part[p], test_gt_dir_by_part[p],
                             test_density_dir_by_part[p], part=p)
    print(f"part_{p} density maps cached at:", train_density_dir_by_part[p],
          "and", test_density_dir_by_part[p])


## 5. Dataset & train/validation split

- **Training** samples are randomly cropped to `patch_size x patch_size` (padding up first if an
  image is smaller) plus a random horizontal flip — classic patch-based augmentation that also
  bounds the memory needed per training step regardless of the source image resolution.
- **Validation/test** samples keep their full resolution, only padded up to a multiple of
  `patch_size` so they can later be split into whole tiles for sliding-window inference.
- The GT density map is downsampled (via **sum-pooling**, which preserves the total count) to
  match the network's output stride, keeping the target tensor — and the loss computation — an
  order of magnitude smaller in memory than a full-resolution target.
- The official ShanghaiTech *training* split is further divided into train/validation here; the
  official *test* split is kept untouched as a final held-out test set.
- With `cfg.part = "both"`, the split above is done **per part first, then combined** (`ConcatDataset`) -- so both Part A and Part B are represented proportionally in train, validation, and test, rather than risking one part being under- or over-sampled.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_to_tensor_norm = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def pad_to_min_size(img_np, density_np, min_h, min_w):
    h, w = img_np.shape[:2]
    pad_h, pad_w = max(0, min_h - h), max(0, min_w - w)
    if pad_h or pad_w:
        img_np = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        density_np = np.pad(density_np, ((0, pad_h), (0, pad_w)), mode="constant")
    return img_np, density_np


def pad_to_multiple(img_np, density_np, multiple):
    h, w = img_np.shape[:2]
    new_h = int(np.ceil(h / multiple) * multiple)
    new_w = int(np.ceil(w / multiple) * multiple)
    pad_h, pad_w = new_h - h, new_w - w
    if pad_h or pad_w:
        img_np = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        density_np = np.pad(density_np, ((0, pad_h), (0, pad_w)), mode="constant")
    return img_np, density_np


def random_crop(img_np, density_np, patch_size):
    h, w = img_np.shape[:2]
    top = random.randint(0, h - patch_size)
    left = random.randint(0, w - patch_size)
    img_c = img_np[top:top + patch_size, left:left + patch_size, :]
    density_c = density_np[top:top + patch_size, left:left + patch_size]
    return img_c, density_c


def sum_pool(density, stride):
    # density: (B, 1, H, W) -> (B, 1, H/stride, W/stride), sum-preserving
    pooled = F.avg_pool2d(density, kernel_size=stride, stride=stride)
    return pooled * (stride * stride)


class ShanghaiTechDataset(Dataset):
    def __init__(self, image_paths, density_dir, patch_size, output_stride,
                 density_scale, train=True):
        self.image_paths = image_paths
        self.density_dir = density_dir
        self.patch_size = patch_size
        self.output_stride = output_stride
        self.density_scale = density_scale
        self.train = train

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        fname = os.path.splitext(os.path.basename(img_path))[0]
        img = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)
        density = load_density_map(self.density_dir, fname).astype(np.float32)

        if self.train:
            img, density = pad_to_min_size(img, density, self.patch_size, self.patch_size)
            img, density = random_crop(img, density, self.patch_size)
            if random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :])
                density = np.ascontiguousarray(density[:, ::-1])
        else:
            img, density = pad_to_multiple(img, density, self.patch_size)

        img_tensor = _to_tensor_norm(img)
        density_tensor = torch.from_numpy(density.copy()).unsqueeze(0).unsqueeze(0).float()
        density_tensor = density_tensor * self.density_scale
        density_ds = sum_pool(density_tensor, self.output_stride).squeeze(0)
        raw_count = torch.tensor(float(density.sum()), dtype=torch.float32)
        return img_tensor, density_ds, raw_count


In [ ]:
def stratified_train_val_split(image_paths, val_split, seed=42):
    idx = list(range(len(image_paths)))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(len(idx) * val_split))
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return [image_paths[i] for i in train_idx], [image_paths[i] for i in val_idx]


train_datasets, val_datasets, test_datasets_by_part = [], [], {}
for p in active_parts:
    part_train_paths, part_val_paths = stratified_train_val_split(
        train_image_paths_by_part[p], cfg.val_split)
    train_datasets.append(ShanghaiTechDataset(
        part_train_paths, train_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=True))
    val_datasets.append(ShanghaiTechDataset(
        part_val_paths, train_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=False))
    test_datasets_by_part[p] = ShanghaiTechDataset(
        test_image_paths_by_part[p], test_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=False)
    print(f"part_{p}: {len(part_train_paths)} train / {len(part_val_paths)} val "
          f"/ {len(test_datasets_by_part[p])} test")

# A single part just falls back to its own dataset; "both" concatenates across parts so the
# rest of the notebook (training loop, diagnostics, visualisation) needs no further changes --
# ConcatDataset supports the same __len__/__getitem__ interface as a plain Dataset.
train_dataset = train_datasets[0] if len(train_datasets) == 1 else ConcatDataset(train_datasets)
val_dataset = val_datasets[0] if len(val_datasets) == 1 else ConcatDataset(val_datasets)
test_dataset = (next(iter(test_datasets_by_part.values())) if len(test_datasets_by_part) == 1
                else ConcatDataset(list(test_datasets_by_part.values())))

print(f"\ntotal: train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True,
                          persistent_workers=cfg.num_workers > 0, drop_last=True)
# Validation/test images keep their native (padded) resolution, which varies per sample,
# so they are loaded one at a time and processed with tiled sliding-window inference.
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)
# Kept separately too so Section 11 can report per-part test metrics alongside the combined one.
test_loaders_by_part = {
    p: DataLoader(ds, batch_size=1, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    for p, ds in test_datasets_by_part.items()
}


## 6. Model

### 6.1 Positional encoding, dilated pyramid pooling, spatial transformer, graph reasoning


In [ ]:
class PositionalEncoding2D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        channels = int(np.ceil(channels / 4) * 2)
        self.channels = channels
        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2).float() / channels))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x):
        b, c, h, w = x.shape
        pos_h = torch.arange(h, device=x.device, dtype=self.inv_freq.dtype)
        pos_w = torch.arange(w, device=x.device, dtype=self.inv_freq.dtype)
        sin_inp_h = torch.einsum("i,j->ij", pos_h, self.inv_freq)
        sin_inp_w = torch.einsum("i,j->ij", pos_w, self.inv_freq)
        emb_h = torch.cat([sin_inp_h.sin(), sin_inp_h.cos()], dim=-1)  # (H, channels)
        emb_w = torch.cat([sin_inp_w.sin(), sin_inp_w.cos()], dim=-1)  # (W, channels)
        emb = torch.zeros(2 * self.channels, h, w, device=x.device, dtype=x.dtype)
        emb[:self.channels] = emb_h.t().unsqueeze(2).expand(-1, -1, w)
        emb[self.channels:2 * self.channels] = emb_w.t().unsqueeze(1).expand(-1, h, -1)
        emb = emb[:c].unsqueeze(0)
        return x + emb


class DilatedPyramidPooling(nn.Module):
    """ASPP-style module: parallel dilated-conv branches at several rates plus a
    global-pooling branch, fused together -- covers both the 'dilated convolutions'
    and 'pyramid structure' multi-scale-context requirements in one block."""

    def __init__(self, in_channels, out_channels, dilations=(1, 2, 3, 4)):
        super().__init__()
        branch_ch = out_channels // (len(dilations) + 1)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, branch_ch, kernel_size=3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(branch_ch),
                nn.ReLU(inplace=True),
            ) for d in dilations
        ])
        self.global_branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, branch_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(branch_ch),
            nn.ReLU(inplace=True),
        )
        total_ch = branch_ch * (len(dilations) + 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [b(x) for b in self.branches]
        g = self.global_branch(x)
        g = F.interpolate(g, size=(h, w), mode="bilinear", align_corners=False)
        feats.append(g)
        return self.fuse(torch.cat(feats, dim=1))


class SpatialTransformer(nn.Module):
    """Learns a per-image affine warp so downstream layers see a perspective/scale
    -normalised feature map -- useful because crowd scenes are shot from many angles
    and distances."""

    def __init__(self, in_channels):
        super().__init__()
        self.loc_net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(16 * 4 * 4, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 6),
        )
        self.fc_loc[-1].weight.data.zero_()
        self.fc_loc[-1].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float32))

    def forward(self, x):
        xs = self.loc_net(x)
        xs = xs.reshape(xs.size(0), -1)
        theta = self.fc_loc(xs).view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode="border")


class TransformerContext(nn.Module):
    """Global self-attention over the (coarse, stride-8) spatial grid so every location
    can attend to crowd context anywhere in the patch."""

    def __init__(self, channels, num_layers=2, num_heads=4, ff_dim=512, dropout=0.1):
        super().__init__()
        self.pos_enc = PositionalEncoding2D(channels)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = self.pos_enc(x).flatten(2).transpose(1, 2)  # (B, H*W, C)
        out = self.transformer(tokens)
        return out.transpose(1, 2).reshape(b, c, h, w)


class GraphOcclusionModule(nn.Module):
    """Graph-based occlusion handling: pools the feature map to a coarse grid of
    'region' nodes, builds a soft top-k similarity graph between them, and propagates
    messages so a heavily-occluded region can borrow evidence from visually-similar,
    less-occluded regions elsewhere in the scene."""

    def __init__(self, channels, grid_size=16, k=8):
        super().__init__()
        self.grid_size = grid_size
        self.k = k
        self.pool = nn.AdaptiveAvgPool2d(grid_size)
        self.node_transform = nn.Linear(channels, channels)
        self.message_transform = nn.Linear(channels, channels)
        self.update = nn.Sequential(nn.Linear(channels * 2, channels), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        b, c, h, w = x.shape
        g = self.grid_size
        nodes = self.pool(x)
        nodes_flat = nodes.flatten(2).transpose(1, 2)  # (B, N, C), N = g*g

        q = self.node_transform(nodes_flat)
        q_norm = F.normalize(q, dim=-1)
        sim = torch.bmm(q_norm, q_norm.transpose(1, 2))  # (B, N, N)

        topk = min(self.k, sim.size(-1))
        topk_val, topk_idx = sim.topk(topk, dim=-1)
        mask = torch.full_like(sim, float("-inf"))
        mask.scatter_(-1, topk_idx, topk_val)
        adj = F.softmax(mask, dim=-1)

        messages = self.message_transform(nodes_flat)
        agg = torch.bmm(adj, messages)
        updated = self.update(torch.cat([nodes_flat, agg], dim=-1))
        updated = updated.transpose(1, 2).reshape(b, c, g, g)

        updated_up = F.interpolate(updated, size=(h, w), mode="bilinear", align_corners=False)
        return self.fuse(torch.cat([x, updated_up], dim=1))


### 6.2 Putting it together

A VGG16 backbone contributes two feature depths (`conv3_3` at stride 4 and `conv4_3` at
stride 8); they are fused into one multi-scale feature map before entering the
dilated-pyramid → spatial-transformer → transformer → graph-reasoning pipeline and a small
convolutional decoder head that regresses the density map.


In [ ]:
class HybridCNNTransformerCrowdCounter(nn.Module):
    def __init__(self, pretrained=True, freeze_stage1=True, transformer_layers=2,
                 transformer_heads=4, transformer_ff_dim=512, graph_grid_size=16,
                 graph_k=8, use_grad_checkpoint=False):
        super().__init__()
        weights = tv_models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None
        vgg_features = tv_models.vgg16(weights=weights).features

        self.stage3 = vgg_features[:16]   # -> stride 4,  256 channels (conv3_3 + relu)
        self.stage4 = vgg_features[16:23]  # -> stride 8,  512 channels (pool3 + conv4 block)

        if freeze_stage1:
            for p in vgg_features[:10].parameters():  # conv1 + conv2 blocks
                p.requires_grad_(False)

        self.lateral3 = nn.Conv2d(256, 256, kernel_size=1)
        self.down3 = nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1)  # stride4 -> stride8
        self.lateral4 = nn.Conv2d(512, 256, kernel_size=1)
        self.fuse_ms = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.dppm = DilatedPyramidPooling(256, 256)
        self.stn = SpatialTransformer(256)
        self.transformer = TransformerContext(256, transformer_layers, transformer_heads,
                                               transformer_ff_dim)
        self.graph = GraphOcclusionModule(256, graph_grid_size, graph_k)

        self.decoder = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU(inplace=True),  # density values must be non-negative
        )

        self.use_grad_checkpoint = use_grad_checkpoint

    def _maybe_checkpoint(self, module, x):
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(module, x, use_reentrant=False)
        return module(x)

    def forward(self, x):
        f3 = self._maybe_checkpoint(self.stage3, x)
        f4 = self._maybe_checkpoint(self.stage4, f3)

        f3_ds = self.down3(self.lateral3(f3))
        f4_p = self.lateral4(f4)
        fused = self.fuse_ms(torch.cat([f3_ds, f4_p], dim=1))

        x = self.dppm(fused)
        x = self.stn(x)
        x = self._maybe_checkpoint(self.transformer, x)
        x = self.graph(x)
        return self.decoder(x)


model = HybridCNNTransformerCrowdCounter(
    pretrained=cfg.pretrained_backbone,
    freeze_stage1=cfg.freeze_backbone_stage1,
    transformer_layers=cfg.transformer_layers,
    transformer_heads=cfg.transformer_heads,
    transformer_ff_dim=cfg.transformer_ff_dim,
    graph_grid_size=cfg.graph_grid_size,
    graph_k=cfg.graph_k,
    use_grad_checkpoint=cfg.use_grad_checkpoint,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_params/1e6:.2f}M | trainable: {n_trainable/1e6:.2f}M")

with torch.no_grad():
    test_out = model(torch.randn(2, 3, cfg.patch_size, cfg.patch_size, device=DEVICE))
print("Output shape for a", cfg.patch_size, "x", cfg.patch_size, "input:", tuple(test_out.shape))


## 7. Loss functions

Three complementary, cheap-to-compute terms combined into one objective:

1. **Pixel-wise MSE** (Euclidean loss) — the classic density-map regression loss.
2. **SSIM loss** — rewards local structural/pattern similarity between predicted and
   ground-truth density maps, which plain MSE ignores (it is a purely point-wise loss).
3. **Direct count L1 loss** — penalises the difference between the *summed* predicted and
   ground-truth counts, directly optimising the metric (MAE) crowd counting is judged on.

`L = lambda_mse * MSE + lambda_ssim * (1 - SSIM) + lambda_count * |count_pred - count_gt|`


In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, channel=1, sigma=1.5):
        super().__init__()
        self.window_size = window_size
        self.channel = channel
        gauss = torch.tensor(
            [math.exp(-((x - window_size // 2) ** 2) / float(2 * sigma ** 2)) for x in range(window_size)]
        )
        gauss = gauss / gauss.sum()
        _2d = gauss.unsqueeze(1) @ gauss.unsqueeze(0)
        window = _2d.float().unsqueeze(0).unsqueeze(0).expand(channel, 1, window_size, window_size)
        self.register_buffer("window", window.contiguous())

    def forward(self, pred, target):
        window = self.window.to(dtype=pred.dtype)
        c1, c2 = 0.01 ** 2, 0.03 ** 2
        pad = self.window_size // 2
        mu1 = F.conv2d(pred, window, padding=pad, groups=self.channel)
        mu2 = F.conv2d(target, window, padding=pad, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sigma1_sq = F.conv2d(pred * pred, window, padding=pad, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(target * target, window, padding=pad, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(pred * target, window, padding=pad, groups=self.channel) - mu1_mu2
        ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / \
                   ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2) + 1e-12)
        return 1 - ssim_map.mean()


class CrowdCountingLoss(nn.Module):
    def __init__(self, lambda_mse=1.0, lambda_ssim=0.1, lambda_count=0.02):
        super().__init__()
        self.lambda_mse = lambda_mse
        self.lambda_ssim = lambda_ssim
        self.lambda_count = lambda_count
        self.mse = nn.MSELoss()
        self.ssim = SSIMLoss()

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        ssim_loss = self.ssim(pred, target)
        pred_count = pred.sum(dim=[1, 2, 3])
        target_count = target.sum(dim=[1, 2, 3])
        count_loss = F.l1_loss(pred_count, target_count)
        total = (self.lambda_mse * mse_loss
                 + self.lambda_ssim * ssim_loss
                 + self.lambda_count * count_loss)
        parts = {
            "mse": mse_loss.item(), "ssim": ssim_loss.item(),
            "count": count_loss.item(), "total": total.item(),
        }
        return total, parts


def compute_mae_mse(pred, target, density_scale):
    pred_count = pred.sum(dim=[1, 2, 3]) / density_scale
    target_count = target.sum(dim=[1, 2, 3]) / density_scale
    mae = (pred_count - target_count).abs()
    se = (pred_count - target_count) ** 2
    return mae, se


criterion = CrowdCountingLoss(cfg.lambda_mse, cfg.lambda_ssim, cfg.lambda_count).to(DEVICE)


## 8. Tiled (sliding-window) inference

Full-resolution test images can be large, and self-attention cost grows quadratically with
token count — feeding a whole image through the transformer at once would spike memory usage.
Instead, `tiled_predict` walks the (already patch-multiple-padded) image in non-overlapping
`patch_size` tiles, runs the model on each tile independently (exactly like training), and
stitches the tile outputs back into one density map. This bounds peak memory to a single
patch's cost no matter how large the source image is, and is reused for validation, test-set
evaluation, and ad-hoc inference on new images.


In [ ]:
@torch.no_grad()
def tiled_predict(model, img, patch_size, output_stride, device, amp=True):
    model.eval()
    img = img.to(device)
    _, _, H, W = img.shape
    out_h, out_w = H // output_stride, W // output_stride
    pred = torch.zeros(1, 1, out_h, out_w, device=device)

    for y in range(0, H, patch_size):
        for x in range(0, W, patch_size):
            patch = img[:, :, y:y + patch_size, x:x + patch_size]
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(amp and device.type == "cuda")):
                p = model(patch)
            oy, ox = y // output_stride, x // output_stride
            pred[:, :, oy:oy + p.shape[-2], ox:ox + p.shape[-1]] = p.float()
    return pred


## 9. Training & validation loops

Mixed precision (`torch.cuda.amp`), optional gradient accumulation, and periodic
`torch.cuda.empty_cache()` calls keep memory usage in check during training.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, cfg, epoch):
    model.train()
    running_loss, running_mae, n_samples, n_skipped = 0.0, 0.0, 0, 0
    optimizer.zero_grad()
    pbar = tqdm(loader, desc=f"Epoch {epoch} [train]", leave=False)
    for step, (imgs, densities, _) in enumerate(pbar):
        imgs = imgs.to(device, non_blocking=True)
        densities = densities.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(cfg.amp and device.type == "cuda")):
            preds = model(imgs)
        # The loss (squares/divisions inside SSIM) is computed in float32 regardless of the
        # autocast dtype used for the forward pass -- squaring already density_scale-scaled
        # values can overflow float16's ~65504 max, which otherwise silently turns into NaN.
        loss, parts = criterion(preds.float(), densities.float())

        if not torch.isfinite(loss):
            print(f"  [warn] non-finite loss at epoch {epoch} step {step}: {parts} -- skipping batch")
            optimizer.zero_grad(set_to_none=True)
            n_skipped += 1
            continue

        loss_scaled = loss / cfg.grad_accum_steps
        scaler.scale(loss_scaled).backward()
        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        with torch.no_grad():
            mae, _ = compute_mae_mse(preds.float(), densities.float(), cfg.density_scale)

        bs = imgs.size(0)
        running_loss += parts["total"] * bs
        running_mae += mae.sum().item()
        n_samples += bs
        pbar.set_postfix(loss=f"{parts['total']:.4f}", mae=f"{mae.mean().item():.2f}",
                          lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    if n_skipped:
        print(f"  Epoch {epoch}: skipped {n_skipped} non-finite batch(es).")
    torch.cuda.empty_cache()
    return running_loss / max(n_samples, 1), running_mae / max(n_samples, 1)


@torch.no_grad()
def validate(model, loader, criterion, device, cfg, desc="Validate"):
    model.eval()
    running_loss, running_mae, running_se, n = 0.0, 0.0, 0.0, 0
    for imgs, densities, _ in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        densities = densities.to(device, non_blocking=True)
        preds = tiled_predict(model, imgs, cfg.patch_size, cfg.output_stride, device, amp=cfg.amp)
        loss, parts = criterion(preds, densities)
        mae, se = compute_mae_mse(preds, densities, cfg.density_scale)

        running_loss += parts["total"] * imgs.size(0)
        running_mae += mae.sum().item()
        running_se += se.sum().item()
        n += imgs.size(0)

    torch.cuda.empty_cache()
    return running_loss / n, running_mae / n, math.sqrt(running_se / n)


## 10. Train

- **Adaptive learning rate**: `ReduceLROnPlateau` halves the LR whenever validation MAE stalls,
  so the schedule adapts to actual training progress rather than following a fixed decay curve.
- **Best-model checkpointing**: after every epoch, the current validation MAE is compared
  against the *best* MAE seen in any previous epoch (a strictly stronger guarantee than only
  checking the immediately preceding epoch) — the checkpoint is overwritten only on improvement.
- **Visibility**: loss and MAE (the regression stand-in for "accuracy" in a counting task) are
  printed and plotted for both the training and validation sets every epoch.
- **Numerical stability**: gradients are clipped to `cfg.grad_clip_norm`, the loss is always
  computed in float32 even under mixed precision, and mixed precision prefers `bfloat16` over
  `float16` when available -- together these prevent the exploding-gradient / fp16-overflow NaNs
  that (once they occur) permanently corrupt the model's weights and `BatchNorm` running stats.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.base_lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=cfg.lr_factor, patience=cfg.lr_patience, min_lr=cfg.lr_min
)
# GradScaler's loss-scaling only matters for float16 (it is a documented no-op when
# enabled=False), so bfloat16 runs simply skip it -- there is nothing for it to protect against.
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": [], "lr": []}
best_val_mae = float("inf")
best_ckpt_path = os.path.join(cfg.ckpt_dir, "best_model.pth")
last_ckpt_path = os.path.join(cfg.ckpt_dir, "last_model.pth")


def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

    axes[1].plot(history["train_mae"], label="train")
    axes[1].plot(history["val_mae"], label="val")
    axes[1].set_title("MAE (count error)"); axes[1].set_xlabel("epoch"); axes[1].legend()

    axes[2].plot(history["lr"])
    axes[2].set_title("Learning rate"); axes[2].set_xlabel("epoch"); axes[2].set_yscale("log")
    plt.tight_layout()
    plt.show()


def run_training(model, optimizer, scheduler, scaler, start_epoch, end_epoch, history, best_val_mae):
    for epoch in range(start_epoch, end_epoch + 1):
        train_loss, train_mae = train_one_epoch(model, train_loader, optimizer, criterion,
                                                 scaler, DEVICE, cfg, epoch)
        val_loss, val_mae, val_rmse = validate(model, val_loader, criterion, DEVICE, cfg)
        scheduler.step(val_mae)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)
        history["lr"].append(current_lr)

        improved = val_mae < best_val_mae
        if improved:
            best_val_mae = val_mae
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(), "val_mae": val_mae,
                "cfg": asdict(cfg),
            }, best_ckpt_path)
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(), "val_mae": val_mae,
        }, last_ckpt_path)

        clear_output(wait=True)
        plot_history(history)
        flag = " ** new best, saved **" if improved else ""
        print(f"Epoch {epoch}/{cfg.epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
              f"train_mae={train_mae:.2f} val_mae={val_mae:.2f} val_rmse={val_rmse:.2f} | "
              f"lr={current_lr:.2e}{flag}")
    return best_val_mae


best_val_mae = run_training(model, optimizer, scheduler, scaler, 1, cfg.epochs, history, best_val_mae)


### 10.1 Resuming after a diverged (NaN) or interrupted run

If `train_loss` / `val_loss` ever print as `nan`, the model's weights -- and, subtly, every
`BatchNorm2d` layer's running statistics, which update during the forward pass itself and are
**not** protected by `GradScaler` -- are permanently corrupted from that point on. No amount of
further training or learning-rate decay recovers them, because NaNs propagate through every
subsequent forward pass. That is why the loss can stay `nan` for the rest of a run even while the
scheduler keeps dutifully reducing the learning rate.

Two things are needed:

1. **Prevention** (already built into `train_one_epoch`/`tiled_predict` above): the loss is
   computed in float32 even under mixed precision (the SSIM term squares and sums already
   `density_scale`-scaled values, which can silently overflow float16's ~65504 max), gradients
   are clipped to `cfg.grad_clip_norm`, and mixed precision prefers `bfloat16` over `float16`
   whenever the GPU supports it (bf16 shares float32's exponent range, so it cannot overflow the
   way float16 can).
2. **Recovery**: reload the last checkpoint saved *before* the divergence. `best_model.pth` is
   only overwritten when validation MAE improves, so — unlike `last_model.pth`, which is
   overwritten every epoch including the NaN one — it is very likely still a healthy,
   pre-divergence checkpoint.

Run the cell below (instead of re-running the cell above) to resume training with the fixes in
effect. It works whether this is the same Colab runtime that diverged or a fresh one.


In [ ]:
resume_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(resume_ckpt["model_state"])
print(f"Restored weights from epoch {resume_ckpt['epoch']} (val_mae={resume_ckpt['val_mae']:.2f})")

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.base_lr, weight_decay=cfg.weight_decay)
optimizer.load_state_dict(resume_ckpt["optimizer_state"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=cfg.lr_factor, patience=cfg.lr_patience, min_lr=cfg.lr_min
)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

try:
    history
except NameError:
    history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": [], "lr": []}

# Drop any epochs recorded after the restored checkpoint (e.g. the NaN epoch itself) so the
# plotted history stays consistent with the restored weights.
resume_epoch = resume_ckpt["epoch"]
for key in history:
    history[key] = history[key][:resume_epoch]
best_val_mae = resume_ckpt["val_mae"]

best_val_mae = run_training(model, optimizer, scheduler, scaler,
                             resume_epoch + 1, cfg.epochs, history, best_val_mae)


### 10.2 Diagnosing the train/val gap

Validation MAE (~500) sitting suspiciously close to Part A's *average* image count, while
training MAE (~75) looks fine, has more than one possible explanation -- rather than guess, this
runs the current best checkpoint on a handful of validation images twice:

1. through the normal **eval-mode BatchNorm** path (`tiled_predict`, using the running
   statistics accumulated during training) -- exactly what `validate()` uses, and
2. with BatchNorm **forced into train mode** on that same tile (using the tile's own batch
   statistics instead of the running average).

- If the **train-mode-BN error is much lower**, the running statistics haven't calibrated well
  for whole-image, eval-mode inference at `batch_size=8` -- the fix is to replace `BatchNorm2d`
  with `GroupNorm` in the custom modules, which doesn't depend on batch statistics at all.
- If **both errors are similarly large**, this is a genuine generalization gap (or a scale
  mismatch between patch-level training MAE and whole-image validation MAE) rather than a
  BatchNorm artifact, and a different fix is needed (more augmentation/regularization, or a
  less aggressively-decayed LR schedule so the model actually finishes converging).


In [ ]:
def summarize_batchnorm_stats(model):
    print(f"{'layer':40s} {'mean(run_mean)':>15s} {'std(run_mean)':>15s} {'mean(run_var)':>15s} {'std(run_var)':>15s}")
    for name, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            rm, rv = m.running_mean, m.running_var
            print(f"{name:40s} {rm.mean().item():15.4f} {rm.std().item():15.4f} "
                  f"{rv.mean().item():15.4f} {rv.std().item():15.4f}")


@torch.no_grad()
def tiled_predict_bn_mode(model, img, patch_size, output_stride, device, amp, bn_train_mode):
    # Identical to tiled_predict, but batches ALL of the image's tiles into one forward pass
    # (instead of one tile at a time) so BatchNorm, if forced into train mode, sees more than a
    # single sample per channel. Needed because DPPM's global-pooling branch collapses each
    # tile to a 1x1 spatial map, and BatchNorm's train-mode variance calculation errors out on
    # a batch of size 1 ("Expected more than 1 value per channel when training").
    model.train(bn_train_mode)
    img = img.to(device)
    _, _, H, W = img.shape
    out_h, out_w = H // output_stride, W // output_stride
    pred = torch.zeros(1, 1, out_h, out_w, device=device)

    patches, positions = [], []
    for y in range(0, H, patch_size):
        for x in range(0, W, patch_size):
            patches.append(img[:, :, y:y + patch_size, x:x + patch_size])
            positions.append((y, x))
    batch = torch.cat(patches, dim=0)
    if bn_train_mode and batch.size(0) == 1:
        batch = batch.repeat(2, 1, 1, 1)  # BatchNorm train mode needs >1 sample per channel

    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(amp and device.type == "cuda")):
        out = model(batch)
    out = out.float()[:len(patches)]

    for (y, x), p in zip(positions, out):
        oy, ox = y // output_stride, x // output_stride
        pred[:, :, oy:oy + p.shape[-2], ox:ox + p.shape[-1]] = p.unsqueeze(0)

    model.eval()
    return pred


diag_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(diag_ckpt["model_state"])
model.eval()
print(f"Diagnosing checkpoint from epoch {diag_ckpt['epoch']} (val_mae={diag_ckpt['val_mae']:.2f})\n")

summarize_batchnorm_stats(model)

n_diag = min(6, len(val_dataset))
diag_idx = random.sample(range(len(val_dataset)), n_diag)

eval_errs, trainbn_errs = [], []
print(f"\n{'idx':>5s} {'gt_count':>10s} {'eval_mode':>10s} {'eval_err':>10s} {'trainBN_mode':>13s} {'trainBN_err':>12s}")
for idx in diag_idx:
    img_tensor, density_gt, _ = val_dataset[idx]
    gt_count = density_gt.sum().item() / cfg.density_scale

    pred_eval = tiled_predict_bn_mode(model, img_tensor.unsqueeze(0), cfg.patch_size,
                                       cfg.output_stride, DEVICE, cfg.amp, bn_train_mode=False)
    pred_trainbn = tiled_predict_bn_mode(model, img_tensor.unsqueeze(0), cfg.patch_size,
                                          cfg.output_stride, DEVICE, cfg.amp, bn_train_mode=True)

    eval_count = pred_eval.sum().item() / cfg.density_scale
    trainbn_count = pred_trainbn.sum().item() / cfg.density_scale
    eval_err = abs(eval_count - gt_count)
    trainbn_err = abs(trainbn_count - gt_count)
    eval_errs.append(eval_err)
    trainbn_errs.append(trainbn_err)

    print(f"{idx:5d} {gt_count:10.1f} {eval_count:10.1f} {eval_err:10.1f} {trainbn_count:13.1f} {trainbn_err:12.1f}")

print(f"\nMean abs error -- eval-mode BN (what validate() uses): {np.mean(eval_errs):.1f}")
print(f"Mean abs error -- train-mode BN (tile's own batch stats): {np.mean(trainbn_errs):.1f}")
print("\nIf train-mode-BN error is dramatically lower -> BatchNorm running stats explain the gap")
print("(switch BatchNorm2d -> GroupNorm). If both are similarly large -> a genuine generalization")
print("gap or patch-vs-whole-image scale mismatch instead; inspect the visualization below.")


In [ ]:
def visualize_bn_diagnosis(model, dataset, idx, cfg, device):
    img_tensor, density_gt, _ = dataset[idx]
    pred_eval = tiled_predict_bn_mode(model, img_tensor.unsqueeze(0), cfg.patch_size,
                                       cfg.output_stride, device, cfg.amp, bn_train_mode=False)
    pred_trainbn = tiled_predict_bn_mode(model, img_tensor.unsqueeze(0), cfg.patch_size,
                                          cfg.output_stride, device, cfg.amp, bn_train_mode=True)

    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_np = np.clip(img_np, 0, 1)

    gt_count = density_gt.sum().item() / cfg.density_scale
    eval_count = pred_eval.sum().item() / cfg.density_scale
    trainbn_count = pred_trainbn.sum().item() / cfg.density_scale

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(img_np); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(density_gt.squeeze(0).numpy(), cmap="jet")
    axes[1].set_title(f"GT (count={gt_count:.1f})"); axes[1].axis("off")
    axes[2].imshow(pred_eval.squeeze().cpu().numpy(), cmap="jet")
    axes[2].set_title(f"Eval-mode BN (count={eval_count:.1f})"); axes[2].axis("off")
    axes[3].imshow(pred_trainbn.squeeze().cpu().numpy(), cmap="jet")
    axes[3].set_title(f"Train-mode BN (count={trainbn_count:.1f})"); axes[3].axis("off")
    plt.tight_layout()
    plt.show()


visualize_bn_diagnosis(model, val_dataset, diag_idx[0], cfg, DEVICE)


## 11. Evaluate the best model on the held-out test set


In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded best model from epoch {ckpt['epoch']} (val_mae={ckpt['val_mae']:.2f})")

test_loss, test_mae, test_rmse = validate(model, test_loader, criterion, DEVICE, cfg, desc="Test")
label = "Test set (combined)" if len(test_loaders_by_part) > 1 else "Test set"
print(f"{label}  ->  loss={test_loss:.4f}  MAE={test_mae:.2f}  RMSE={test_rmse:.2f}")

if len(test_loaders_by_part) > 1:
    for p, loader in test_loaders_by_part.items():
        p_loss, p_mae, p_rmse = validate(model, loader, criterion, DEVICE, cfg, desc=f"Test part_{p}")
        print(f"  part_{p}  ->  loss={p_loss:.4f}  MAE={p_mae:.2f}  RMSE={p_rmse:.2f}")


## 12. Qualitative visualisation

Compare a few test images against their ground-truth and predicted density maps (with
predicted vs. actual head counts).


In [ ]:
def visualize_samples(model, dataset, cfg, device, n=3):
    idxs = random.sample(range(len(dataset)), min(n, len(dataset)))
    fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 4 * len(idxs)))
    if len(idxs) == 1:
        axes = axes[None, :]

    for row, idx in enumerate(idxs):
        img_tensor, density_gt, _ = dataset[idx]
        pred = tiled_predict(model, img_tensor.unsqueeze(0), cfg.patch_size,
                              cfg.output_stride, device, amp=cfg.amp)

        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)

        gt_count = density_gt.sum().item() / cfg.density_scale
        pred_count = pred.sum().item() / cfg.density_scale

        axes[row, 0].imshow(img_np); axes[row, 0].set_title("Input"); axes[row, 0].axis("off")
        axes[row, 1].imshow(density_gt.squeeze(0).numpy(), cmap="jet")
        axes[row, 1].set_title(f"GT density (count={gt_count:.1f})"); axes[row, 1].axis("off")
        axes[row, 2].imshow(pred.squeeze().cpu().numpy(), cmap="jet")
        axes[row, 2].set_title(f"Predicted density (count={pred_count:.1f})"); axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()


visualize_samples(model, test_dataset, cfg, DEVICE, n=3)


## 13. Inference on a new image

Memory-efficient, patch-based inference for any arbitrary image (not necessarily from
ShanghaiTech).


In [ ]:
def predict_count(model, image_path, cfg, device):
    img = np.array(Image.open(image_path).convert("RGB"), dtype=np.uint8)
    h, w = img.shape[:2]
    new_h = int(np.ceil(h / cfg.patch_size) * cfg.patch_size)
    new_w = int(np.ceil(w / cfg.patch_size) * cfg.patch_size)
    img_padded = np.pad(img, ((0, new_h - h), (0, new_w - w), (0, 0)), mode="reflect")

    img_tensor = _to_tensor_norm(img_padded).unsqueeze(0)
    pred_density = tiled_predict(model, img_tensor, cfg.patch_size, cfg.output_stride,
                                  device, amp=cfg.amp)
    count = pred_density.sum().item() / cfg.density_scale
    return count, pred_density.squeeze().cpu().numpy()


# Example:
# count, density_map = predict_count(model, "/content/some_crowd_photo.jpg", cfg, DEVICE)
# print(f"Estimated crowd count: {count:.1f}")


## 14. Memory-optimisation techniques used in this notebook

- **Patch-based training** (`patch_size`-sized random crops) and **tiled sliding-window
  inference** (`tiled_predict`) — bound peak activation memory (including the O(N²)
  self-attention cost) to a single patch regardless of source image resolution.
- **Automatic mixed precision** (`torch.autocast` + `GradScaler`) — halves activation memory
  and speeds up matmuls on modern GPUs.
- **Gradient accumulation** (`grad_accum_steps`) — simulates a larger effective batch size
  without holding all samples' activations at once.
- **Partial backbone freezing** (`freeze_backbone_stage1`) — no gradients/optimizer state are
  kept for the frozen early VGG layers.
- **Optional gradient checkpointing** (`use_grad_checkpoint`) — trades recompute for memory on
  the backbone/transformer if you hit OOM on a smaller GPU.
- **Sum-pooled, downsampled density targets** — the loss/metrics operate on a `1/output_stride`
  resolution target instead of a full-resolution one, shrinking both target and prediction
  tensors substantially.
- **Disk-cached density maps** (`.npy` per image) — the expensive Gaussian-splat step runs once,
  not every epoch.
- **DataLoader tuning** — `pin_memory`, `persistent_workers`, and modest `num_workers` overlap
  host-side preprocessing with GPU compute without duplicating worker startup cost each epoch.
- Periodic `torch.cuda.empty_cache()` after each epoch's train/val pass.
- **Numerically-stable mixed precision** (`bfloat16` when available, float32 loss computation, gradient clipping) -- prevents the NaN corruption described in Section 10.1, which otherwise silently wastes an entire run's compute.

## 15. (Optional) Persist results to Google Drive


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# drive_ckpt_dir = "/content/drive/MyDrive/crowd_counting_checkpoints"
# os.makedirs(drive_ckpt_dir, exist_ok=True)
# shutil.copy(best_ckpt_path, os.path.join(drive_ckpt_dir, "best_model.pth"))
# print("Saved best model to Google Drive.")
